
# 03 — GDP Recovery Dynamic Baseline

This notebook constructs the GDP recovery outcome used later for validation.

It runs after:

```text
02_Raw_Files_Coverage_Diagnostics_2002_5Var.ipynb
```

## Final methodological decisions reflected here

- The source coverage starts from **2002**.
- The final POSet ordering variables are **5**, not 6.
- WGI/governance is **not** part of the final POSet ordering.
- GDP recovery is **not** an ordering variable.
- GDP recovery is used only as an external validation/outcome variable.

## Recovery logic

The recovery outcome measures how many years a country takes to return to its pre-shock GDP level after a negative growth shock.

Because the actual negative-growth year can differ by country, the baseline is dynamic:

### 2007 shock

Check 2008 and 2009:

- if first negative growth is in 2008 → baseline year = 2007
- if first negative growth is in 2009 → baseline year = 2008
- if no negative growth in 2008/2009 → recovery value = 0
- if no shock-window data → review/remove

### 2019 shock

Check 2020 and 2021:

- if first negative growth is in 2020 → baseline year = 2019
- if first negative growth is in 2021 → baseline year = 2020
- if no negative growth in 2020/2021 → recovery value = 0
- if no shock-window data → review/remove

## Special values

| Situation | Recovery value |
|---|---:|
| No negative growth in shock window | 0 |
| Not recovered by last available year | 20 |
| No shock-window data | missing / review |

## Main outputs

- `country_recovery_summary_dynamic_baseline_2007_2019.csv`
- `country_recovery_audit_dynamic_baseline.csv`
- `country_recovery_index_paths_dynamic_baseline.csv`
- `country_recovery_status_summary_dynamic_baseline.csv`
- `recovery_problem_cases.csv`


In [1]:

# ------------------------------------------------------
# IMPORTS AND PATHS
# ------------------------------------------------------

from pathlib import Path
from datetime import datetime
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 250)
pd.set_option("display.float_format", "{:.4f}".format)

PROJECT_ROOT = Path.cwd()

INPUT_DIR = PROJECT_ROOT / "Data" / "Processed" / "Comparable_Raw_Files"

PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed" / "GDP_Recovery_Dynamic_Baseline"
DIAGNOSTICS_DIR = PROJECT_ROOT / "Data" / "Diagnostics" / "03_GDP_Recovery_Dynamic_Baseline"
TABLES_DIR = PROJECT_ROOT / "Outputs" / "Tables" / "GDP_Recovery_Dynamic_Baseline"
AUDIT_DIR = PROJECT_ROOT / "Data" / "Audit"

for folder in [PROCESSED_DIR, DIAGNOSTICS_DIR, TABLES_DIR, AUDIT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

START_YEAR = 2002
END_YEAR = 2025

print("Run ID:", RUN_ID)
print("Project root:", PROJECT_ROOT.resolve())
print("Input folder:", INPUT_DIR.resolve())
print("Processed folder:", PROCESSED_DIR.resolve())


Run ID: 20260624_182721
Project root: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24
Input folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\Comparable_Raw_Files
Processed folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\GDP_Recovery_Dynamic_Baseline


In [2]:

# ------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------

STANDARDIZED_LONG_CANDIDATES = [
    PROJECT_ROOT / "Data" / "Processed" / "Comparable_Raw_Files" / "standardized_country_year_variable_long.csv",
    PROJECT_ROOT / "Data" / "Processed" / "Comparable_Raw_Files" / "standardized_all_variables_long.csv",
]

def find_first_existing_path(candidates, label):
    for path in candidates:
        if path.exists():
            print(f"Using {label}: {path}")
            return path
    raise FileNotFoundError(
        f"Could not find {label}. Tried:\n" +
        "\n".join([f"- {p}" for p in candidates])
    )

STANDARDIZED_LONG_FILE = find_first_existing_path(
    STANDARDIZED_LONG_CANDIDATES,
    "standardized comparable long dataset",
)

GDP_GROWTH_VARIABLE = "gdp_growth"

SHOCK_CONFIGS = {
    "2007": {
        "shock_label": "Global Financial Crisis",
        "shock_window": [2008, 2009],
        "not_recovered_value": 20,
    },
    "2019": {
        "shock_label": "COVID shock",
        "shock_window": [2020, 2021],
        "not_recovered_value": 20,
    },
}

print("GDP growth variable:", GDP_GROWTH_VARIABLE)
display(pd.DataFrame([
    {
        "shock_id": k,
        "shock_label": v["shock_label"],
        "shock_window": ",".join(map(str, v["shock_window"])),
        "not_recovered_value": v["not_recovered_value"],
    }
    for k, v in SHOCK_CONFIGS.items()
]))


Using standardized comparable long dataset: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\Comparable_Raw_Files\standardized_country_year_variable_long.csv
GDP growth variable: gdp_growth


,shock_id,shock_label,shock_window,not_recovered_value
0,2007,Global Financial Crisis,"2008,2009",20
1,2019,COVID shock,"2020,2021",20


In [3]:

# ------------------------------------------------------
# LOAD GDP GROWTH DATA
# ------------------------------------------------------

df = pd.read_csv(STANDARDIZED_LONG_FILE)

# Normalize possible column names defensively.
rename_candidates = {
    "country": "Country",
    "iso3": "Country",
    "ISO3": "Country",
    "year": "Year",
    "Variable": "variable",
    "variable_name": "variable",
    "value": "Value",
    "obs_value": "Value",
    "OBS_VALUE": "Value",
}

for old_col, new_col in rename_candidates.items():
    if old_col in df.columns and new_col not in df.columns:
        df = df.rename(columns={old_col: new_col})

required_cols = {"Country", "Year", "variable", "Value"}
missing_cols = required_cols - set(df.columns)

if missing_cols:
    raise ValueError(
        f"Input file missing required columns: {missing_cols}\n"
        f"Selected file: {STANDARDIZED_LONG_FILE}\n"
        f"Available columns: {list(df.columns)}"
    )

df["Country"] = df["Country"].astype(str).str.strip().str.upper()
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
df["Value"] = pd.to_numeric(df["Value"], errors="coerce")
df["variable"] = df["variable"].astype(str).str.strip()

df = (
    df.dropna(subset=["Country", "Year", "variable", "Value"])
    .assign(Year=lambda d: d["Year"].astype(int))
    .query("Year >= @START_YEAR and Year <= @END_YEAR")
    .copy()
)

gdp = df[df["variable"].eq(GDP_GROWTH_VARIABLE)].copy()

if gdp.empty:
    raise ValueError(f"No rows found for variable: {GDP_GROWTH_VARIABLE}")

# Defensive country-code filter.
invalid_country_codes = gdp[~gdp["Country"].str.match(r"^[A-Z]{3}$", na=False)].copy()
invalid_country_codes.to_csv(DIAGNOSTICS_DIR / "invalid_country_codes_in_gdp_growth.csv", index=False)

if len(invalid_country_codes) > 0:
    raise ValueError("Invalid country codes found in GDP growth data.")

gdp = (
    gdp[["Country", "Year", "Value"]]
    .rename(columns={"Value": "gdp_growth"})
    .drop_duplicates(["Country", "Year"])
    .sort_values(["Country", "Year"])
    .reset_index(drop=True)
)

print("GDP growth rows:", len(gdp))
print("Countries:", gdp["Country"].nunique())
print("Years:", gdp["Year"].min(), "-", gdp["Year"].max())
display(gdp.head())


GDP growth rows: 1043
Countries: 44
Years: 2002 - 2025


,Country,Year,gdp_growth
0,AUS,2002,3.0913
1,AUS,2003,4.2189
2,AUS,2004,3.1555
3,AUS,2005,2.7709
4,AUS,2006,3.7895


In [4]:

# ------------------------------------------------------
# SAVE HELPER
# ------------------------------------------------------

table_inventory_rows = []

def save_table(table, file_name, title, description):
    processed_path = PROCESSED_DIR / file_name
    diagnostics_path = DIAGNOSTICS_DIR / file_name
    table_path = TABLES_DIR / file_name

    table.to_csv(processed_path, index=False)
    table.to_csv(diagnostics_path, index=False)
    table.to_csv(table_path, index=False)

    table_inventory_rows.append({
        "file_name": file_name,
        "title": title,
        "description": description,
        "rows": len(table),
        "columns": len(table.columns),
        "processed_path": str(processed_path),
        "diagnostics_path": str(diagnostics_path),
        "table_path": str(table_path),
        "created_at": RUN_TIMESTAMP,
    })

    print("Saved:", file_name)


In [5]:

# ------------------------------------------------------
# INPUT COVERAGE CHECK
# ------------------------------------------------------

gdp_growth_recovery_input_coverage = (
    gdp
    .groupby("Country")
    .agg(
        min_year=("Year", "min"),
        max_year=("Year", "max"),
        years_available=("Year", "nunique"),
        min_growth=("gdp_growth", "min"),
        median_growth=("gdp_growth", "median"),
        max_growth=("gdp_growth", "max"),
    )
    .reset_index()
    .sort_values("Country")
)

# Add shock-window availability flags.
for shock_id, cfg in SHOCK_CONFIGS.items():
    window = cfg["shock_window"]
    window_col = f"has_shock_window_data_{shock_id}"
    temp = (
        gdp[gdp["Year"].isin(window)]
        .groupby("Country")["Year"]
        .nunique()
        .reset_index(name=f"shock_window_years_available_{shock_id}")
    )
    gdp_growth_recovery_input_coverage = gdp_growth_recovery_input_coverage.merge(temp, on="Country", how="left")
    gdp_growth_recovery_input_coverage[f"shock_window_years_available_{shock_id}"] = gdp_growth_recovery_input_coverage[f"shock_window_years_available_{shock_id}"].fillna(0).astype(int)
    gdp_growth_recovery_input_coverage[window_col] = gdp_growth_recovery_input_coverage[f"shock_window_years_available_{shock_id}"].gt(0)

save_table(
    gdp_growth_recovery_input_coverage,
    "gdp_growth_recovery_input_coverage.csv",
    "GDP growth recovery input coverage",
    "Country-level coverage diagnostics for GDP growth recovery construction.",
)

display(gdp_growth_recovery_input_coverage.head())


Saved: gdp_growth_recovery_input_coverage.csv


,Country,min_year,max_year,years_available,min_growth,median_growth,max_growth,shock_window_years_available_2007,has_shock_window_data_2007,shock_window_years_available_2019,has_shock_window_data_2019
0,AUS,2002,2025,24,-0.1335,2.6278,4.2530,2,True,2,True
1,AUT,2002,2025,24,-6.3183,1.6197,5.3310,2,True,2,True
2,BEL,2002,2025,24,-4.7930,1.6341,6.2545,2,True,2,True
3,BRA,2002,2023,22,-3.5458,3.0108,7.5282,2,True,2,True
4,CAN,2002,2025,24,-5.0382,2.1879,5.9505,2,True,2,True


In [6]:

# ------------------------------------------------------
# RECOVERY CALCULATION FUNCTION
# ------------------------------------------------------

def calculate_dynamic_recovery_for_country(country, country_df, shock_id, cfg):
    country_df = country_df.sort_values("Year").copy()
    growth_by_year = dict(zip(country_df["Year"], country_df["gdp_growth"]))

    shock_window = cfg["shock_window"]
    not_recovered_value = cfg["not_recovered_value"]

    shock_values = {
        year: growth_by_year.get(year, np.nan)
        for year in shock_window
    }

    available_shock_values = {
        year: value
        for year, value in shock_values.items()
        if pd.notna(value)
    }

    if len(available_shock_values) == 0:
        return {
            "Country": country,
            "shock_id": shock_id,
            "shock_label": cfg["shock_label"],
            "status": "no_shock_window_data",
            "first_negative_year": np.nan,
            "dynamic_baseline_year": np.nan,
            "baseline_gdp_index": np.nan,
            "recovery_year": np.nan,
            "recovery_years": np.nan,
            "final_available_year": country_df["Year"].max() if len(country_df) else np.nan,
            "final_gdp_index": np.nan,
            "not_recovered_value_used": np.nan,
            "shock_window_growth_values": str(shock_values),
            "note": "No GDP growth value available in the shock-window years.",
        }

    first_negative_year = None
    for year in shock_window:
        value = growth_by_year.get(year, np.nan)
        if pd.notna(value) and value < 0:
            first_negative_year = year
            break

    if first_negative_year is None:
        return {
            "Country": country,
            "shock_id": shock_id,
            "shock_label": cfg["shock_label"],
            "status": "no_negative_growth_in_shock_window",
            "first_negative_year": np.nan,
            "dynamic_baseline_year": min(available_shock_values.keys()) - 1,
            "baseline_gdp_index": 100.0,
            "recovery_year": min(available_shock_values.keys()),
            "recovery_years": 0,
            "final_available_year": country_df["Year"].max() if len(country_df) else np.nan,
            "final_gdp_index": np.nan,
            "not_recovered_value_used": 0,
            "shock_window_growth_values": str(shock_values),
            "note": "No negative GDP growth observed in shock window; assigned recovery value 0.",
        }

    dynamic_baseline_year = first_negative_year - 1

    if dynamic_baseline_year not in growth_by_year and dynamic_baseline_year < country_df["Year"].min():
        return {
            "Country": country,
            "shock_id": shock_id,
            "shock_label": cfg["shock_label"],
            "status": "missing_dynamic_baseline_context",
            "first_negative_year": first_negative_year,
            "dynamic_baseline_year": dynamic_baseline_year,
            "baseline_gdp_index": np.nan,
            "recovery_year": np.nan,
            "recovery_years": np.nan,
            "final_available_year": country_df["Year"].max() if len(country_df) else np.nan,
            "final_gdp_index": np.nan,
            "not_recovered_value_used": np.nan,
            "shock_window_growth_values": str(shock_values),
            "note": "Dynamic baseline year is outside available GDP growth context.",
        }

    # Build GDP index path from dynamic baseline year onward.
    path_rows = []
    index = 100.0

    path_rows.append({
        "Country": country,
        "shock_id": shock_id,
        "Year": dynamic_baseline_year,
        "gdp_growth": np.nan,
        "gdp_index_dynamic_baseline_100": index,
        "dynamic_baseline_year": dynamic_baseline_year,
        "first_negative_year": first_negative_year,
    })

    recovery_year = None

    for year in sorted([y for y in growth_by_year if y > dynamic_baseline_year]):
        growth = growth_by_year[year]

        if pd.isna(growth):
            continue

        index = index * (1 + growth / 100.0)

        path_rows.append({
            "Country": country,
            "shock_id": shock_id,
            "Year": year,
            "gdp_growth": growth,
            "gdp_index_dynamic_baseline_100": index,
            "dynamic_baseline_year": dynamic_baseline_year,
            "first_negative_year": first_negative_year,
        })

        if year >= first_negative_year and recovery_year is None and index >= 100.0:
            recovery_year = year

    if len(path_rows) > 0:
        final_available_year = max([r["Year"] for r in path_rows])
        final_gdp_index = path_rows[-1]["gdp_index_dynamic_baseline_100"]
    else:
        final_available_year = np.nan
        final_gdp_index = np.nan

    if recovery_year is None:
        status = "not_recovered_by_available_data"
        recovery_years = not_recovered_value
        recovery_year = np.nan
        note = f"GDP index did not return to 100 by last available year; assigned {not_recovered_value}."
    else:
        status = "recovered"
        recovery_years = recovery_year - first_negative_year
        note = "GDP index returned to dynamic pre-shock baseline."

    return {
        "Country": country,
        "shock_id": shock_id,
        "shock_label": cfg["shock_label"],
        "status": status,
        "first_negative_year": first_negative_year,
        "dynamic_baseline_year": dynamic_baseline_year,
        "baseline_gdp_index": 100.0,
        "recovery_year": recovery_year,
        "recovery_years": recovery_years,
        "final_available_year": final_available_year,
        "final_gdp_index": final_gdp_index,
        "not_recovered_value_used": not_recovered_value if status == "not_recovered_by_available_data" else np.nan,
        "shock_window_growth_values": str(shock_values),
        "note": note,
    }


def build_index_path_for_country(country, country_df, shock_id, cfg):
    result = calculate_dynamic_recovery_for_country(country, country_df, shock_id, cfg)

    if pd.isna(result["dynamic_baseline_year"]) or pd.isna(result["first_negative_year"]):
        return pd.DataFrame()

    country_df = country_df.sort_values("Year").copy()
    growth_by_year = dict(zip(country_df["Year"], country_df["gdp_growth"]))

    dynamic_baseline_year = int(result["dynamic_baseline_year"])
    first_negative_year = int(result["first_negative_year"])

    path_rows = []
    index = 100.0

    path_rows.append({
        "Country": country,
        "shock_id": shock_id,
        "Year": dynamic_baseline_year,
        "gdp_growth": np.nan,
        "gdp_index_dynamic_baseline_100": index,
        "dynamic_baseline_year": dynamic_baseline_year,
        "first_negative_year": first_negative_year,
    })

    for year in sorted([y for y in growth_by_year if y > dynamic_baseline_year]):
        growth = growth_by_year[year]
        if pd.isna(growth):
            continue

        index = index * (1 + growth / 100.0)

        path_rows.append({
            "Country": country,
            "shock_id": shock_id,
            "Year": year,
            "gdp_growth": growth,
            "gdp_index_dynamic_baseline_100": index,
            "dynamic_baseline_year": dynamic_baseline_year,
            "first_negative_year": first_negative_year,
        })

    return pd.DataFrame(path_rows)


In [7]:

# ------------------------------------------------------
# RUN RECOVERY CALCULATION
# ------------------------------------------------------

summary_rows = []
path_frames = []

for shock_id, cfg in SHOCK_CONFIGS.items():
    print("=" * 90)
    print("Processing shock:", shock_id, cfg["shock_label"])

    for country, country_df in gdp.groupby("Country"):
        result = calculate_dynamic_recovery_for_country(country, country_df, shock_id, cfg)
        summary_rows.append(result)

        path_df = build_index_path_for_country(country, country_df, shock_id, cfg)
        if len(path_df) > 0:
            path_frames.append(path_df)

country_recovery_audit_dynamic_baseline = pd.DataFrame(summary_rows)

if path_frames:
    country_recovery_index_paths_dynamic_baseline = pd.concat(path_frames, ignore_index=True)
else:
    country_recovery_index_paths_dynamic_baseline = pd.DataFrame()

# Wide summary for easy merge later.
country_recovery_summary_dynamic_baseline_2007_2019 = (
    country_recovery_audit_dynamic_baseline
    .pivot_table(
        index="Country",
        columns="shock_id",
        values="recovery_years",
        aggfunc="first",
    )
    .reset_index()
)

country_recovery_summary_dynamic_baseline_2007_2019 = country_recovery_summary_dynamic_baseline_2007_2019.rename(columns={
    "2007": "Recovery_2007",
    "2019": "Recovery_2019",
})

# Add status columns too.
status_wide = (
    country_recovery_audit_dynamic_baseline
    .pivot_table(
        index="Country",
        columns="shock_id",
        values="status",
        aggfunc="first",
    )
    .reset_index()
    .rename(columns={
        "2007": "Recovery_2007_status",
        "2019": "Recovery_2019_status",
    })
)

country_recovery_summary_dynamic_baseline_2007_2019 = country_recovery_summary_dynamic_baseline_2007_2019.merge(
    status_wide,
    on="Country",
    how="left",
)

save_table(
    country_recovery_audit_dynamic_baseline,
    "country_recovery_audit_dynamic_baseline.csv",
    "GDP recovery audit dynamic baseline",
    "Long-format country-shock recovery audit with dynamic baseline choices and status.",
)

save_table(
    country_recovery_summary_dynamic_baseline_2007_2019,
    "country_recovery_summary_dynamic_baseline_2007_2019.csv",
    "GDP recovery summary dynamic baseline",
    "Wide-format recovery values for 2007 and 2019 shocks, ready for master dataset merge.",
)

save_table(
    country_recovery_index_paths_dynamic_baseline,
    "country_recovery_index_paths_dynamic_baseline.csv",
    "GDP recovery index paths dynamic baseline",
    "Country-shock-year GDP index paths normalized to dynamic baseline = 100.",
)

display(country_recovery_audit_dynamic_baseline.head())
display(country_recovery_summary_dynamic_baseline_2007_2019.head())


Processing shock: 2007 Global Financial Crisis
Processing shock: 2019 COVID shock
Saved: country_recovery_audit_dynamic_baseline.csv
Saved: country_recovery_summary_dynamic_baseline_2007_2019.csv
Saved: country_recovery_index_paths_dynamic_baseline.csv


,Country,shock_id,shock_label,status,first_negative_year,dynamic_baseline_year,baseline_gdp_index,recovery_year,recovery_years,final_available_year,final_gdp_index,not_recovered_value_used,shock_window_growth_values,note
0,AUS,2007,Global Financial Crisis,no_negative_growth_in_shock_window,NaN,2007.0000,100.0000,2008.0000,0.0000,2025,NaN,0.0000,"{2008: 1.95484826298521, 2009: 2.21004795300734}",No negative GDP growth observed in shock windo...
1,AUT,2007,Global Financial Crisis,recovered,2009.0000,2008.0000,100.0000,2011.0000,2.0000,2025,116.0124,NaN,"{2008: 1.45330580608418, 2009: -3.58626483259779}",GDP index returned to dynamic pre-shock baseline.
2,BEL,2007,Global Financial Crisis,recovered,2009.0000,2008.0000,100.0000,2010.0000,1.0000,2025,124.6582,NaN,"{2008: 0.446905130727231, 2009: -1.90644105515...",GDP index returned to dynamic pre-shock baseline.
3,BRA,2007,Global Financial Crisis,recovered,2009.0000,2008.0000,100.0000,2010.0000,1.0000,2023,123.6974,NaN,"{2008: 5.09419555783932, 2009: -0.125811937286...",GDP index returned to dynamic pre-shock baseline.
4,CAN,2007,Global Financial Crisis,recovered,2009.0000,2008.0000,100.0000,2010.0000,1.0000,2025,135.4034,NaN,"{2008: 0.995406342461433, 2009: -2.91508625839...",GDP index returned to dynamic pre-shock baseline.


shock_id,Country,Recovery_2007,Recovery_2019,Recovery_2007_status,Recovery_2019_status
0,AUS,0.0000,0.0000,no_negative_growth_in_shock_window,no_negative_growth_in_shock_window
1,AUT,2.0000,2.0000,recovered,recovered
2,BEL,1.0000,1.0000,recovered,recovered
3,BRA,1.0000,1.0000,recovered,recovered
4,CAN,1.0000,1.0000,recovered,recovered


In [8]:

# ------------------------------------------------------
# STATUS SUMMARY AND PROBLEM CASES
# ------------------------------------------------------

country_recovery_status_summary_dynamic_baseline = (
    country_recovery_audit_dynamic_baseline
    .groupby(["shock_id", "status"])
    .agg(countries=("Country", "nunique"))
    .reset_index()
    .sort_values(["shock_id", "status"])
)

problem_statuses = {
    "no_shock_window_data",
    "missing_dynamic_baseline_context",
    "not_recovered_by_available_data",
}

recovery_problem_cases = country_recovery_audit_dynamic_baseline[
    country_recovery_audit_dynamic_baseline["status"].isin(problem_statuses)
].copy()

save_table(
    country_recovery_status_summary_dynamic_baseline,
    "country_recovery_status_summary_dynamic_baseline.csv",
    "GDP recovery status summary",
    "Counts of recovery status categories by shock.",
)

save_table(
    recovery_problem_cases,
    "recovery_problem_cases.csv",
    "GDP recovery problem cases",
    "Countries needing review because of missing data or not recovered status.",
)

display(country_recovery_status_summary_dynamic_baseline)
display(recovery_problem_cases.head(100))


Saved: country_recovery_status_summary_dynamic_baseline.csv
Saved: recovery_problem_cases.csv


,shock_id,status,countries
0,2007,no_negative_growth_in_shock_window,8
1,2007,not_recovered_by_available_data,1
2,2007,recovered,35
3,2019,no_negative_growth_in_shock_window,6
4,2019,no_shock_window_data,1
5,2019,recovered,37


,Country,shock_id,shock_label,status,first_negative_year,dynamic_baseline_year,baseline_gdp_index,recovery_year,recovery_years,final_available_year,final_gdp_index,not_recovered_value_used,shock_window_growth_values,note
18,GRC,2007,Global Financial Crisis,not_recovered_by_available_data,2009.0000,2008.0000,100.0000,NaN,20.0000,2025,86.1397,20.0000,"{2008: 0.057473865625609, 2009: -4.11927611877...",GDP index did not return to 100 by last availa...
81,RUS,2019,COVID shock,no_shock_window_data,NaN,NaN,NaN,NaN,NaN,2019,NaN,NaN,"{2020: nan, 2021: nan}",No GDP growth value available in the shock-win...


In [9]:

# ------------------------------------------------------
# MANUAL VALIDATION CASES
# ------------------------------------------------------
# ARG has been useful as a sanity check in earlier versions.

VALIDATION_COUNTRIES = ["ARG", "DEU", "ITA", "USA"]

manual_validation_cases = country_recovery_audit_dynamic_baseline[
    country_recovery_audit_dynamic_baseline["Country"].isin(VALIDATION_COUNTRIES)
].copy()

manual_validation_growth_values = gdp[
    gdp["Country"].isin(VALIDATION_COUNTRIES)
    & gdp["Year"].between(2007, 2024)
].copy()

save_table(
    manual_validation_cases,
    "manual_validation_recovery_cases.csv",
    "Manual validation recovery cases",
    "Selected country recovery audit rows for manual inspection.",
)

save_table(
    manual_validation_growth_values,
    "manual_validation_gdp_growth_values.csv",
    "Manual validation GDP growth values",
    "GDP growth values for selected countries used as manual sanity checks.",
)

display(manual_validation_cases)
display(manual_validation_growth_values.head(80))


Saved: manual_validation_recovery_cases.csv
Saved: manual_validation_gdp_growth_values.csv


,Country,shock_id,shock_label,status,first_negative_year,dynamic_baseline_year,baseline_gdp_index,recovery_year,recovery_years,final_available_year,final_gdp_index,not_recovered_value_used,shock_window_growth_values,note
11,DEU,2007,Global Financial Crisis,recovered,2009.0000,2008.0000,100.0000,2011.0000,2.0000,2025,115.0495,NaN,"{2008: 0.887902330743618, 2009: -5.54455451920...",GDP index returned to dynamic pre-shock baseline.
25,ITA,2007,Global Financial Crisis,recovered,2008.0000,2007.0000,100.0000,2023.0000,15.0000,2025,101.9040,NaN,"{2008: -1.02313851177097, 2009: -5.30515401233...",GDP index returned to dynamic pre-shock baseline.
42,USA,2007,Global Financial Crisis,recovered,2009.0000,2008.0000,100.0000,2010.0000,1.0000,2025,142.2002,NaN,"{2008: 0.113587248160993, 2009: -2.57650023225...",GDP index returned to dynamic pre-shock baseline.
55,DEU,2019,COVID shock,recovered,2020.0000,2019.0000,100.0000,2022.0000,2.0000,2025,100.2588,NaN,"{2020: -4.13191443239948, 2021: 3.90999994204108}",GDP index returned to dynamic pre-shock baseline.
69,ITA,2019,COVID shock,recovered,2020.0000,2019.0000,100.0000,2022.0000,2.0000,2025,106.4105,NaN,"{2020: -8.86822122185344, 2021: 8.93106210800055}",GDP index returned to dynamic pre-shock baseline.
86,USA,2019,COVID shock,recovered,2020.0000,2019.0000,100.0000,2021.0000,1.0000,2025,115.1944,NaN,"{2020: -2.08137597860093, 2021: 6.15202248021889}",GDP index returned to dynamic pre-shock baseline.


,Country,Year,gdp_growth
264,DEU,2007,2.8891
265,DEU,2008,0.8879
266,DEU,2009,-5.5446
267,DEU,2010,4.1346
268,DEU,2011,3.7580
269,DEU,2012,0.4635
270,DEU,2013,0.3970
271,DEU,2014,2.1802
272,DEU,2015,1.6630
273,DEU,2016,2.2222


In [10]:

# ------------------------------------------------------
# AUDIT WORKBOOK
# ------------------------------------------------------

audit_xlsx = AUDIT_DIR / "03_GDP_Recovery_Dynamic_Baseline_Audit.xlsx"

with pd.ExcelWriter(audit_xlsx, engine="xlsxwriter") as writer:
    gdp_growth_recovery_input_coverage.to_excel(writer, sheet_name="input_coverage", index=False)
    country_recovery_status_summary_dynamic_baseline.to_excel(writer, sheet_name="status_summary", index=False)
    country_recovery_audit_dynamic_baseline.to_excel(writer, sheet_name="recovery_audit", index=False)
    country_recovery_summary_dynamic_baseline_2007_2019.to_excel(writer, sheet_name="recovery_summary_wide", index=False)
    recovery_problem_cases.to_excel(writer, sheet_name="problem_cases", index=False)
    manual_validation_cases.to_excel(writer, sheet_name="manual_cases", index=False)
    manual_validation_growth_values.to_excel(writer, sheet_name="manual_growth_values", index=False)
    pd.DataFrame(table_inventory_rows).to_excel(writer, sheet_name="table_inventory", index=False)

print("Audit workbook saved:", audit_xlsx.resolve())


Audit workbook saved: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Audit\03_GDP_Recovery_Dynamic_Baseline_Audit.xlsx


In [11]:

# ------------------------------------------------------
# FINAL COMPLETION CHECK
# ------------------------------------------------------

table_inventory = pd.DataFrame(table_inventory_rows)
table_inventory.to_csv(PROCESSED_DIR / "gdp_recovery_table_inventory.csv", index=False)
table_inventory.to_csv(DIAGNOSTICS_DIR / "gdp_recovery_table_inventory.csv", index=False)

expected_outputs = [
    "gdp_growth_recovery_input_coverage.csv",
    "country_recovery_audit_dynamic_baseline.csv",
    "country_recovery_summary_dynamic_baseline_2007_2019.csv",
    "country_recovery_index_paths_dynamic_baseline.csv",
    "country_recovery_status_summary_dynamic_baseline.csv",
    "recovery_problem_cases.csv",
    "manual_validation_recovery_cases.csv",
    "manual_validation_gdp_growth_values.csv",
]

output_check = pd.DataFrame([
    {
        "file_name": f,
        "processed_exists": (PROCESSED_DIR / f).exists(),
        "diagnostics_exists": (DIAGNOSTICS_DIR / f).exists(),
        "tables_exists": (TABLES_DIR / f).exists(),
    }
    for f in expected_outputs
])

output_check.to_csv(DIAGNOSTICS_DIR / "output_check.csv", index=False)

print("03 GDP RECOVERY DYNAMIC BASELINE COMPLETE")
print("=" * 90)
display(output_check)

print("\nKey outputs to inspect/send:")
print("- country_recovery_status_summary_dynamic_baseline.csv")
print("- country_recovery_summary_dynamic_baseline_2007_2019.csv")
print("- recovery_problem_cases.csv")
print("- 03_GDP_Recovery_Dynamic_Baseline_Audit.xlsx")

print("\nNext notebook:")
print("04_WGI_Governance_Compilation_2002_ContextOnly.ipynb")


03 GDP RECOVERY DYNAMIC BASELINE COMPLETE


,file_name,processed_exists,diagnostics_exists,tables_exists
0,gdp_growth_recovery_input_coverage.csv,True,True,True
1,country_recovery_audit_dynamic_baseline.csv,True,True,True
2,country_recovery_summary_dynamic_baseline_2007...,True,True,True
3,country_recovery_index_paths_dynamic_baseline.csv,True,True,True
4,country_recovery_status_summary_dynamic_baseli...,True,True,True
5,recovery_problem_cases.csv,True,True,True
6,manual_validation_recovery_cases.csv,True,True,True
7,manual_validation_gdp_growth_values.csv,True,True,True



Key outputs to inspect/send:
- country_recovery_status_summary_dynamic_baseline.csv
- country_recovery_summary_dynamic_baseline_2007_2019.csv
- recovery_problem_cases.csv
- 03_GDP_Recovery_Dynamic_Baseline_Audit.xlsx

Next notebook:
04_WGI_Governance_Compilation_2002_ContextOnly.ipynb
